# 04 · Matplotlib: กราฟภูมิอากาศสำหรับงานภูมิศาสตร์
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apichayaice7835-png/Teach-Python-Geography/blob/main/04_Matplotlib_Climate_Graphs.ipynb)

**เวลา:** 2–3 ชั่วโมง | **ระดับ:** 🟢 → 🟡 → 🔴

| กราฟ | ใช้ตอบคำถาม |
|---|---|
| Line plot | ฝนเปลี่ยนแปลงตามเดือนอย่างไร |
| Bar chart | สถานีไหนฝนมาก/น้อย |
| **Climograph** | ลักษณะภูมิอากาศของสถานี (ฝน + อุณหภูมิ) |
| Boxplot | ความผันแปรของฝนแต่ละเดือน |
| Small multiples | เปรียบเทียบหลายสถานีพร้อมกัน |
| Time series + rolling mean | แนวโน้มระยะยาว |

**โครงสร้างของกราฟ Matplotlib:** `fig` (กระดาษทั้งแผ่น) → `ax` (แกนกราฟแต่ละช่อง)

In [ ]:
# ============================================================
# ตั้งค่าฟอนต์ภาษาไทยให้ Matplotlib (ถ้าไม่ตั้ง ตัวอักษรไทยจะเป็นสี่เหลี่ยม □□□)
# ============================================================
import glob, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

def setup_thai_font():
    fonts = glob.glob("/usr/share/fonts/**/Loma*.[ot]tf", recursive=True)
    if not fonts:   # บน Colab ต้องติดตั้งก่อน (ใช้เวลา ~10 วินาที ครั้งเดียวต่อ session)
        subprocess.run("apt-get -qq install -y fonts-thai-tlwg > /dev/null", shell=True)
        fonts = glob.glob("/usr/share/fonts/**/Loma*.[ot]tf", recursive=True)
    for f in fonts:
        fm.fontManager.addfont(f)
    if fonts:
        plt.rcParams["font.family"] = "Loma"
        print("✅ ใช้ฟอนต์ไทย Loma")
    else:
        print("⚠️ ติดตั้งฟอนต์ไทยไม่ได้ ใช้ฟอนต์เริ่มต้นแทน")
    plt.rcParams["axes.unicode_minus"] = False   # ให้เครื่องหมายลบแสดงถูกต้อง

setup_thai_font()

In [ ]:
# ============================================================
# เซลล์โหลดข้อมูล (ใช้ซ้ำทุก Notebook)
# ลำดับการหา: 1) โฟลเดอร์ data/ ในเครื่อง  2) GitHub ของอาจารย์
# ============================================================
import os
import pandas as pd

GITHUB_USER = "apichayaice7835-png"      # ← ชื่อบัญชี GitHub ของอาจารย์
REPO        = "Teach-Python-Geography"   # ← ชื่อ repository
BASE_URL    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main/data/"

def data_path(filename):
    """คืน path ของไฟล์: ถ้ามีในเครื่องใช้ในเครื่อง ถ้าไม่มีใช้ลิงก์ GitHub"""
    for folder in ["data", f"{REPO}/data", "/content/data"]:
        p = os.path.join(folder, filename)
        if os.path.exists(p):
            return p
    return BASE_URL + filename

print("ไฟล์สถานี :", data_path("rain_stations_north.csv"))
print("ไฟล์ฝนรายเดือน:", data_path("rain_monthly_2020_2024.csv"))

In [ ]:
import numpy as np
stations = pd.read_csv(data_path("rain_stations_north.csv"))
rain = pd.read_csv(data_path("rain_monthly_2020_2024.csv"), parse_dates=["date"])
rain["year"], rain["month"] = rain["date"].dt.year, rain["date"].dt.month
df = rain.merge(stations, on="station_id")

MONTHS_TH = ["ม.ค.","ก.พ.","มี.ค.","เม.ย.","พ.ค.","มิ.ย.","ก.ค.","ส.ค.","ก.ย.","ต.ค.","พ.ย.","ธ.ค."]
# ค่าเฉลี่ยรายเดือน 5 ปีของแต่ละสถานี (skipna เป็นค่าเริ่มต้น → ค่าว่างไม่ถูกนับ)
clim = df.groupby(["name_th", "month"])[["rain_mm", "tmean_c"]].mean().reset_index()
clim.head()

## 🟢 1. Line plot: ฝนรายเดือนของสถานีเดียว

In [ ]:
cm = clim[clim["name_th"] == "เชียงใหม่"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(MONTHS_TH, cm["rain_mm"], marker="o", color="tab:blue", linewidth=2)
ax.set_title("ปริมาณฝนเฉลี่ยรายเดือน สถานีเชียงใหม่ (2020–2024)")
ax.set_xlabel("เดือน"); ax.set_ylabel("ปริมาณฝน (มม.)")
ax.grid(alpha=0.3)
plt.show()

## 🟢 2. Bar chart: เปรียบเทียบฝนรายปีระหว่างสถานี

In [ ]:
annual = df.groupby(["name_th", "year"])["rain_mm"].sum().groupby("name_th").mean().sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(annual.index, annual.values, color="steelblue")
ax.axvline(annual.mean(), color="red", ls="--", label=f"ค่าเฉลี่ย {annual.mean():.0f} มม.")
ax.bar_label(bars, fmt="%.0f", padding=3, fontsize=8)      # ใส่ตัวเลขท้ายแท่ง
ax.set_xlabel("ฝนรายปีเฉลี่ย (มม.)"); ax.set_title("ฝนรายปีเฉลี่ย 2020–2024 รายสถานี")
ax.legend(); plt.tight_layout(); plt.show()

## 🟡 3. Climograph (กราฟภูมิอากาศ): แท่ง = ฝน, เส้น = อุณหภูมิ
ใช้ `ax.twinx()` สร้างแกน y ที่สองทางขวา

In [ ]:
def climograph(station_name, ax=None):
    """วาด climograph ของสถานีที่กำหนด คืนค่า ax เพื่อใช้ต่อได้"""
    d = clim[clim["name_th"] == station_name]
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(MONTHS_TH, d["rain_mm"], color="#4a90d9", label="ฝน (มม.)")
    ax.set_ylabel("ปริมาณฝน (มม.)", color="#4a90d9")
    ax.set_ylim(0, 400)

    ax2 = ax.twinx()                                   # แกน y ที่สอง
    ax2.plot(MONTHS_TH, d["tmean_c"], color="#d9534f", marker="o", label="อุณหภูมิ (°C)")
    ax2.set_ylabel("อุณหภูมิเฉลี่ย (°C)", color="#d9534f")
    ax2.set_ylim(10, 35)

    elev = stations.loc[stations["name_th"] == station_name, "elev_m"].iloc[0]
    ax.set_title(f"Climograph: {station_name} (สูง {elev} ม.)  ฝนรวม {d['rain_mm'].sum():.0f} มม./ปี")
    return ax

climograph("เชียงใหม่"); plt.show()
climograph("ดอยอ่างขาง"); plt.show()      # เปรียบเทียบสถานีบนภูเขาสูง

## 🟡 4. Boxplot: ความผันแปรของฝนแต่ละเดือน (ทุกสถานี ทุกปี)

In [ ]:
data_by_month = [df.loc[df["month"] == m, "rain_mm"].dropna() for m in range(1, 13)]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.boxplot(data_by_month, patch_artist=True, boxprops=dict(facecolor="lightblue"))
ax.set_xticks(range(1, 13), MONTHS_TH)
ax.set_ylabel("ปริมาณฝนรายเดือน (มม.)")
ax.set_title("การกระจายของฝนรายเดือน 16 สถานี ภาคเหนือ (2020–2024)")
ax.grid(axis="y", alpha=0.3); plt.show()

## 🔴 5. Small multiples: climograph ทุกสถานีในรูปเดียว

In [ ]:
names = stations.sort_values("lat", ascending=False)["name_th"].tolist()    # เรียงจากเหนือลงใต้
fig, axes = plt.subplots(4, 4, figsize=(16, 12), sharex=True)

for ax, name in zip(axes.flat, names):
    d = clim[clim["name_th"] == name]
    ax.bar(range(1, 13), d["rain_mm"], color="#4a90d9")
    ax.set_ylim(0, 450)
    ax2 = ax.twinx(); ax2.plot(range(1, 13), d["tmean_c"], "r-o", ms=3); ax2.set_ylim(10, 35)
    ax.set_title(f"{name} ({d['rain_mm'].sum():.0f} มม.)", fontsize=10)
    ax.set_xticks([1, 4, 7, 10]); ax.set_xticklabels(["ม.ค.", "เม.ย.", "ก.ค.", "ต.ค."])

fig.suptitle("Climograph สถานีภาคเหนือ (เรียงจากเหนือลงใต้)", fontsize=16)
plt.tight_layout(); plt.show()

## 🔴 6. Time series ต่อเนื่อง 60 เดือน + ค่าเฉลี่ยเคลื่อนที่ 12 เดือน

In [ ]:
ts = df[df["station_id"] == "S13"].set_index("date")["rain_mm"]      # สถานีน่าน
roll = ts.rolling(12, center=True, min_periods=9).mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(ts.index, ts.values, alpha=0.3, color="tab:blue", label="ฝนรายเดือน")
ax.plot(roll.index, roll.values, color="navy", lw=2, label="ค่าเฉลี่ยเคลื่อนที่ 12 เดือน")
ax.set_ylabel("มม."); ax.set_title("อนุกรมเวลาปริมาณฝน สถานีน่าน 2020–2024")
ax.legend(loc="upper left"); ax.grid(alpha=0.3)

os.makedirs("output", exist_ok=True)
fig.savefig("output/nan_rain_timeseries.png", dpi=300, bbox_inches="tight")   # dpi 300 สำหรับรายงาน
plt.show()

## 🔴 7. Scatter + เส้นถดถอย: ความสูง vs ฝนรายปี

In [ ]:
st = stations.set_index("name_th").join(annual.rename("annual_mm"))
slope, intercept = np.polyfit(st["elev_m"], st["annual_mm"], 1)     # เส้นตรง y = ax + b
r = np.corrcoef(st["elev_m"], st["annual_mm"])[0, 1]

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(st["elev_m"], st["annual_mm"], c=st["lon"], cmap="viridis", s=70, edgecolor="k")
xx = np.linspace(0, 1500, 50)
ax.plot(xx, slope*xx + intercept, "r--", label=f"y = {slope:.2f}x + {intercept:.0f}  (r = {r:.2f})")
for name, row in st.iterrows():
    ax.annotate(name, (row["elev_m"], row["annual_mm"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
fig.colorbar(sc, label="ลองจิจูด (°E)")
ax.set_xlabel("ความสูง (ม.)"); ax.set_ylabel("ฝนรายปีเฉลี่ย (มม.)"); ax.legend()
plt.show()

---
## ✏️ แบบฝึกหัด
**🟢** 1) วาด line plot ของ 3 สถานีในกราฟเดียว พร้อม legend  2) เปลี่ยนสีและขนาด marker

**🟡** 3) วาด climograph ของสถานีใดก็ได้ แล้วเขียนตีความ 3–5 บรรทัด (ฤดูฝนเริ่ม/สิ้นสุดเมื่อใด เดือนที่ร้อนที่สุด)
4) วาด bar chart แบบ grouped: ฝนรายปี 2020–2024 ของจังหวัดน่าน เทียบเชียงใหม่

**🔴** 5) สร้าง heatmap (`ax.imshow`) ของ anomaly รายเดือน (แถว = ปี-เดือน, คอลัมน์ = สถานี)
6) ปรับกราฟข้อ 7 ให้แสดง 95% confidence band ของเส้นถดถอย